<a href="https://colab.research.google.com/github/simongalaxy/sentimental_agent/blob/master/Amazon_Reviews_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
! pip install sentence-transformers transformers bertopic

In [91]:
import pandas as pd
from typing import List, Dict
from pprint import pformat

In [92]:
filePath = "/content/amazon_google_reviews.csv"

In [ ]:
def load_clean_split_data_from_csv(path: str) -> List[pd.DataFrame]:

  # load data from csv file.
  df = pd.read_csv(path)
  print(df.info())

  # clean data.
  str_cols = ["Platform", "Product/Location_Name", "Review_Text"]
  df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

  # split data into a list of dataframes by platforms and products.
  dfs = []
  for platform in df["Platform"].unique():
    for product in df[df["Platform"] == platform]["Product/Location_Name"].unique():
      df_temp = df[(df["Platform"] == platform) & (df["Product/Location_Name"] == product)].reset_index()
      dfs.append(df_temp)

  print(f"Total {len(dfs)} dataframes splitted.")

  return dfs

In [94]:
dfs = load_clean_split_data_from_csv(path = filePath)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6200 entries, 0 to 6199
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Review_ID              6200 non-null   object
 1   Platform               6200 non-null   object
 2   Product/Location_Name  6200 non-null   object
 3   Review_Text            6200 non-null   object
 4   Rating                 6200 non-null   int64 
 5   Date                   6200 non-null   object
 6   Helpful_Votes          6200 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 339.2+ KB
None
Total 25 dataframes splitted.


In [95]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import paraphrase_mining

# --- Assumed Setup ---
# df = pd.DataFrame({"Review_Text": [...]})
class DuplicatesRemover:
  # 1. Load a lightweight, high-performance embedding model
  def __init__(self):
    self.model_name = "all-MiniLM-L6-v2"
    self.model = SentenceTransformer(self.model_name)

  def _group_semantic_similar_reviews(self, df: pd.DataFrame) -> pd.DataFrame:
    # 1. Extract reviews to a list
    sentences = df["Review_Text"].tolist()

    # 2. Perform Paraphrase Mining
    # (Removed the invalid 'min_score' argument)
    pairs = paraphrase_mining(self.model, sentences)

    # 3. Group the sentences using a Union-Find (Connected Components) structure
    parent = {i: i for i in range(len(sentences))}

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])  # Path compression
        return parent[i]

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    # Merge sentences ONLY if they meet or exceed the similarity threshold
    threshold = 0.85
    for score, i, j in pairs:
        if score >= threshold:
            union(i, j)

    # 4. Map the cluster root IDs directly back to your DataFrame
    # This gives every review in df a Group ID based on its semantic cluster root
    df["Semantic_Group_ID"] = [find(i) for i in range(len(sentences))]

    # 5. Extract the groupings for visual confirmation
    groupings = {}
    for i in range(len(sentences)):
        root = find(i)
        if root not in groupings:
            groupings[root] = []
        groupings[root].append(sentences[i])

    # 6. Print the Results
    print("--- SEMANTIC GROUPINGS ---")
    deduplicated_sentences = []

    for group_id, group in groupings.items():
        print(f"\nGroup {group_id}:")
        for sentence in group:
            print(f" - {sentence}")

        # The representative sentence of this group is the first one
        deduplicated_sentences.append(group[0])

    print("\n--- FINAL DEDUPLICATED LIST ---")
    for sent in deduplicated_sentences:
        print(f"✔ {sent}")

    return df



  def _consolidate_reviews_by_groups(self, df: pd.DataFrame) -> pd.DataFrame:
    # consolidate the df by Semantic_Group_ID.
    df_grouped = df.groupby(by=["Platform", "Product/Location_Name", "Semantic_Group_ID"]).agg(Count=("Review_ID", "count"), Average_Rating=("Rating", "mean"), Helpful_Votes=("Helpful_Votes", "sum")).reset_index()

    # get the first sentences by Semantic_Group_IDs.
    dfs_first_sentence = []
    for id in df["Semantic_Group_ID"].unique():
      temp = df[df["Semantic_Group_ID"] == id][["Semantic_Group_ID", "Review_Text"]].head(1)
      dfs_first_sentence.append(temp)

    df_first_sentence = pd.concat(dfs_first_sentence)

    # merge these 2 dataframes as output.
    df_unique_review = pd.merge(df_grouped, df_first_sentence, on="Semantic_Group_ID")

    return df_unique_review


  def _save_unique_reviews_to_csv(self, dfs: List[pd.DataFrame]) -> None:
    for df in dfs:
      platform = df["Platform"].to_list()[0].replace(" ", "_")
      product = df["Product/Location_Name"].to_list()[0].replace(" ", "_")
      filename = f"{platform}-{product}-unique_review.csv"
      df[["Platform", "Product/Location_Name", "Semantic_Group_ID", "Review_Text", "Count", "Average_Rating", "Helpful_Votes"]].sort_values(by="Review_Text", ascending=True).to_csv(filename)

    return

  def remove_duplicate_reviews(self, dfs: List[pd.DataFrame]) -> List[pd.DataFrame]:
    dfs_processed = [self._group_semantic_similar_reviews(df=df) for df in dfs]
    dfs_unique_review = [self._consolidate_reviews_by_groups(df=df) for df in dfs_processed]
    # self._save_unique_reviews_to_csv(dfs=dfs_unique_review)

    return dfs_unique_review



In [96]:
remover = DuplicatesRemover()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [97]:
dfs_unique_review = remover.remove_duplicate_reviews(dfs=dfs)

Streaming output truncated to the last 5000 lines.
Group 60:
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Summary: Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Bought this recently. Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Absolute perfection! Easily the best purchase

In [98]:
dfs_unique_review[0].head(10)

,Platform,Product/Location_Name,Semantic_Group_ID,Count,Average_Rating,Helpful_Votes,Review_Text
0,Google Locations,Metropolitan Dental Health Clinic,0,2,1.0,2,Terrible hygiene and filthy tables. Saw flies ...
1,Google Locations,Metropolitan Dental Health Clinic,2,10,4.0,11,Wonderful experience overall. The staff was co...
2,Google Locations,Metropolitan Dental Health Clinic,3,4,2.0,8,Mediocre stay. Cleanliness was questionable an...
3,Google Locations,Metropolitan Dental Health Clinic,6,11,5.0,66,"Incredible place to visit! Breathtaking flora,..."
4,Google Locations,Metropolitan Dental Health Clinic,7,7,4.0,11,Great atmosphere and delicious food! The pasta...
5,Google Locations,Metropolitan Dental Health Clinic,9,2,1.0,4,Rude management and hostile attitude towards c...
6,Google Locations,Metropolitan Dental Health Clinic,15,6,3.0,8,"Honestly, decent spot, nothing extraordinary. ..."
7,Google Locations,Metropolitan Dental Health Clinic,18,3,1.0,39,"Honestly, rude management and hostile attitude..."
8,Google Locations,Metropolitan Dental Health Clinic,19,5,3.0,23,Fair experience. Room was adequately clean and...
9,Google Locations,Metropolitan Dental Health Clinic,20,2,1.0,5,Bought this recently. Total rip-off. Paid prem...


In [ ]:
from transformers import pipeline

class SentimentalClassifier:
  def __init__(self):
    # sentimental classifier settings.
    self.sentimental_model = "distilbert-base-uncased-finetuned-sst-2-english"
    self.sentimental_classifier = pipeline("sentiment-analysis", model=self.sentimental_model)

  # Batch process unique reviews
  def label_sentimental(self, dfs: List[pd.DataFrame], review_col: str) -> List[pd.DataFrame]:
    for i, df in enumerate(dfs):
      platform = df["Platform"].to_list()[0].replace(" ", "_")
      product = df["Product/Location_Name"].to_list()[0].replace(" ", "_")
      filename = f"{platform}-{product}-sentimental.csv"

      print(f"No. {i}/{len(dfs)}: Start label sentimental for dataframe - Platform: {platform}, Product: {product}")
      df['Sentiment'] = [res['label'] for res in self.sentimental_classifier(df[review_col].tolist(), batch_size=32)]
      df.to_csv(filename)
      print(f"No. {i}/{len(dfs)}: End label sentimental for dataframe - Platform: {platform}, Product: {product} and save as csv.")

    print(f"total {len(dfs)} dataframes labelled sentimentals.")

    return dfs



In [ ]:
classifier = SentimentalClassifier()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [101]:
dfs_sentimental = classifier.label_sentimental(dfs=dfs_unique_review, review_col="Review_Text")

No. 0/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Metropolitan_Dental_Health_Clinic
No. 0/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Metropolitan_Dental_Health_Clinic and save as csv.
No. 1/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Blue_Bottle_Artisan_Coffee_Bar
No. 1/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Blue_Bottle_Artisan_Coffee_Bar and save as csv.
No. 2/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Bella_Italia_Trattoria
No. 2/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Bella_Italia_Trattoria and save as csv.
No. 3/25: Start label sentimental for dataframe - Platform: Google_Locations, Product: Highland_Public_Library_&_Cultural_Center
No. 3/25: End label sentimental for dataframe - Platform: Google_Locations, Product: Highland_Public_Library_&_Cultural_Center a

In [102]:
dfs_sentimental[24].head(10)

,Platform,Product/Location_Name,Semantic_Group_ID,Count,Average_Rating,Helpful_Votes,Review_Text,Sentiment
0,Amazon,Anker 65W GaN Fast Charger,2,16,5.0,62,Review update after 3 months: Superb product! ...,POSITIVE
1,Amazon,Anker 65W GaN Fast Charger,3,3,2.0,5,"Honestly, looks nice on the desk, but function...",NEGATIVE
2,Amazon,Anker 65W GaN Fast Charger,4,13,5.0,103,Game changer! Saves me so much time and effort...,POSITIVE
3,Amazon,Anker 65W GaN Fast Charger,6,12,5.0,106,Review update after 3 months: Absolute perfect...,POSITIVE
4,Amazon,Anker 65W GaN Fast Charger,8,2,2.0,4,Disappointed with the build quality. It is oka...,NEGATIVE
5,Amazon,Anker 65W GaN Fast Charger,13,6,4.0,8,Impressive performance for the price. Clear so...,POSITIVE
6,Amazon,Anker 65W GaN Fast Charger,15,3,2.0,6,Review update after 3 months: Subpar durabilit...,NEGATIVE
7,Amazon,Anker 65W GaN Fast Charger,16,6,1.0,36,Review update after 3 months: Do not buy this!...,NEGATIVE
8,Amazon,Anker 65W GaN Fast Charger,17,9,4.0,14,Bought this recently. Much better than my prev...,NEGATIVE
9,Amazon,Anker 65W GaN Fast Charger,18,17,5.0,152,Remarkable craftsmanship. Every detail feels t...,POSITIVE


In [103]:
df_sentimental = pd.concat(dfs_sentimental).sort_values(by=["Platform", "Product/Location_Name", "Sentiment"], ascending=[True, True, True]).reset_index()

In [104]:
# from typing import List
# from bertopic import BERTopic
# from bertopic.representation import TextGeneration
# from transformers import pipeline
# from sentence_transformers import SentenceTransformer

# class TopicIdentifier:
#     def __init__(self):
#         # 1. Embedding Model Setup
#         self.embedding_model_name = "all-MiniLM-L6-v2"
#         self.embedding_model = SentenceTransformer(self.embedding_model_name)

#         # 2. Generator Setup (Changed to text2text-generation for Flan-T5)
#         self.generator_model_name = 'google/flan-t5-base'
#         self.generator = pipeline('text-generation', model=self.generator_model_name)

#         # Define a clean prompt so Flan-T5 knows how to summarize the keywords into a label
#         prompt = "I have a topic that contains the following documents: [DOCUMENTS] and keywords: [KEYWORDS]. Based on this, what is the short category name for this topic?"
#         self.representation_model = TextGeneration(self.generator, prompt=prompt)

#         # 3. Initialize BERTopic with built-in outlier cleaning (nr_topics="auto")
#         self.topic_model = BERTopic(
#             embedding_model=self.embedding_model,
#             representation_model=self.representation_model,
#             nr_topics="auto" # Merges tiny/insignificant clusters together automatically
#         )

#     def get_topics_from_reviews(self, reviews: List[str]) -> List[str]:
#         # Train the model
#         topics, probs = self.topic_model.fit_transform(reviews)

#         # Safety Step: Force reduce the outlier (-1) cluster so you don't hit an IndexError later
#         new_topics = self.topic_model.reduce_outliers(reviews, topics, strategy="embeddings")
#         self.topic_model.update_topics(reviews, topics=new_topics)

#         # Extract all topics dictionary {topic_id: [(word, score), ...]}
#         all_topics = self.topic_model.get_topics()

#         # Safely extract ONLY the words for Topic 0 (or whichever valid topic is found first)
#         word_list = []
#         target_topic = 0 if 0 in all_topics else list(all_topics.keys())[0]

#         if target_topic != -1: # Ensure we aren't looking at outliers
#             words_with_scores = all_topics[target_topic]
#             word_list = [word for word, score in words_with_scores]
#             print(f"Topic {target_topic} Words:", word_list)
#         else:
#             print("No distinct topics found. All reviews classified as outliers.")

#         return word_list


In [105]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer


class TopicIdentifier:
  def __init__(self):
    # 2. Configure Sub-modules for deterministic and cleaner results
    # Setting random_state ensures you get the exact same categories every run
    self.umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, random_state=42)
    self.hdbscan_model = HDBSCAN(min_cluster_size=2, metric='euclidean', cluster_selection_method='eom')

    # Ignore generic English stop words so they don't corrupt your category names
    self.vectorizer_model = CountVectorizer(stop_words="english")

    # 3. Choose a powerful Hugging Face embedding model
    # 'all-mpnet-base-v2' offers higher accuracy for nuanced text than 'all-MiniLM-L6-v2'
    self.embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
    self.embedding_model = SentenceTransformer(self.embedding_model_name)

    # 4. Initialize BERTopic with control over the number of final categories
    self.topic_model = BERTopic(
        embedding_model=self.embedding_model,
        umap_model=self.umap_model,
        hdbscan_model=self.hdbscan_model,
        vectorizer_model=self.vectorizer_model,
        nr_topics="auto" # Dynamically merges highly similar categories together
        )

  def get_categories(self, reviews: List[str]):
    # 1. Run the pipeline
    topics, probs = self.topic_model.fit_transform(reviews)

    # 2. Reduce outliers (Optional but highly recommended)
    # This forces reviews grouped into 'Topic -1' (Noise) into their closest real category
    new_topics = self.topic_model.reduce_outliers(reviews, topics, strategy="embeddings")
    self.topic_model.update_topics(reviews, topics=new_topics)

    # 3. Map categories back to your original data
    df_results = pd.DataFrame({
        "Review": reviews,
        "Category_ID": new_topics
    })

    # Get category names
    topic_info = self.topic_model.get_topic_info()
    category_mapping = dict(zip(topic_info['Topic'], topic_info['Name']))
    df_results['Category_Name'] = df_results['Category_ID'].map(category_mapping)

    # 8. View the final categorized data
    print("\n=== DISCOVERED CATEGORIES OVERVIEW ===")
    print(topic_info[["Topic", "Count", "Name"]])

    print("\n=== SAMPLE MAPPED REVIEWS ===")
    print(df_results[["Review", "Category_Name"]].head())

    return


In [106]:
identifier = TopicIdentifier()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [107]:
identifier.get_categories(reviews=df_sentimental["Review_Text"])

2026-10-27 09:26:28,737 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



=== DISCOVERED CATEGORIES OVERVIEW ===
    Topic  Count                                    Name
0       0     40              0_falls_muted_social_media
1       1     35  1_okay_extraordinary_neighborhood_spot
2       2     32                    2_staff_ac_damp_back
3       3     33         3_answered_doctor_gentle_nurses
4       4     29               4_fast_sound_durable_most
..    ...    ...                                     ...
75     75      4               75_20_rest_vanished_water
76     76      3   76_hostile_attitude_customers_towards
77     77      3                   77_10_gem_toast_beans
78     78      3              78_arrive_cold_burnt_steak
79     79      3           79_elevated_dishes_level_meal

[80 rows x 3 columns]

=== SAMPLE MAPPED REVIEWS ===
                                              Review  \
0  Honestly, looks nice on the desk, but function...   
1  Disappointed with the build quality. It is oka...   
2  Review update after 3 months: Subpar durabilit...  

In [108]:
# test_df = dfs_sentimental[10]

# for sentiment in test_df["Sentiment"].unique():
#   print(f"Process {sentiment} reviews.")
#   reviews = test_df[test_df["Sentiment"] == sentiment]["Review_Text"].tolist()
#   print(f"{sentiment} reviews: {reviews}")
#   categories = identifier.get_categories(reviews=reviews)
#   print(f"{sentiment} categories: {categories}\n")

